DATA ENGINEER (DE) DELIVERABLE
================================
Role: Acquires and validates raw audio, extracts audio features,
      generates the processed feature dataset, and performs
      preprocessing/data-quality validation.

Outputs (written to ./processed/):
  - features.csv                  (processed feature dataset)
  - data_quality_log.txt          (data quality audit)
  - corrupt_files.csv             (created only if files fail extraction)

Feature extraction:
  - 40 MFCC features
  - 12 Chroma features
  - 128 Mel-spectrogram features
  - Total: 180 audio features per recording

Dataset validation:
  - 1,440 audio files processed
  - 0 corrupt/skipped files
  - 0 missing feature values
  - 0 infinite feature values
  - 0 duplicate rows
  - 0 duplicate file paths
  - 24 unique actors

The resulting features.csv contains 1,440 rows and 183 columns
(3 metadata columns + 180 extracted audio features).

Train/validation splitting and feature scaling are performed
separately downstream and are not part of this preprocessing stage.

In [18]:
# ============================================================
# RAVDESS DATA PREPROCESSING
# Purpose:
# 1. Read raw RAVDESS audio files
# 2. Extract metadata from filenames
# 3. Extract audio features
# 4. Save processed features as CSV
# 5. Check preprocessing/data quality
# ============================================================

import os
import glob
import warnings

import numpy as np
import pandas as pd
import librosa

from tqdm import tqdm

warnings.filterwarnings("ignore")

In [19]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

DATA_PATH = r"D:\SJU\ML assn\Audio_Speech_Actors_01-24_16k"
OUTPUT_PATH = r"D:\SJU\ML assn\CallConnect\data\processed"

os.makedirs(OUTPUT_PATH, exist_ok=True)

In [20]:
# ============================================================
# 2. EMOTION MAPPING
# ============================================================

EMOTION_MAP = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fearful",
    "07": "disgust",
    "08": "surprised"
}

In [21]:
# ============================================================
# 3. FEATURE EXTRACTION
# ============================================================

def extract_features(file_name):
    """
    Load an audio file and extract numerical audio features.

    Features extracted:
    - 40 MFCC features
    - 12 Chroma features
    - 128 Mel-spectrogram features

    Returns:
        numpy array containing all extracted features.
    """

    # Load audio file
    X, sample_rate = librosa.load(file_name, sr=None)

    # Short-Time Fourier Transform
    stft = np.abs(librosa.stft(X))

    # --------------------------------------------------------
    # MFCC
    # --------------------------------------------------------

    mfccs = np.mean(
        librosa.feature.mfcc(
            y=X,
            sr=sample_rate,
            n_mfcc=40
        ).T,
        axis=0
    )

    # --------------------------------------------------------
    # Chroma
    # --------------------------------------------------------

    chroma = np.mean(
        librosa.feature.chroma_stft(
            S=stft,
            sr=sample_rate
        ).T,
        axis=0
    )

    # --------------------------------------------------------
    # Mel-Spectrogram
    # --------------------------------------------------------

    mel = np.mean(
        librosa.feature.melspectrogram(
            y=X,
            sr=sample_rate
        ).T,
        axis=0
    )

    # Combine all features into one feature vector
    result = np.hstack((mfccs, chroma, mel))

    return result


def get_feature_names():
    """
    Generate column names corresponding to the extracted features.
    """

    return (
        [f"mfcc_{i}" for i in range(40)]
        + [f"chroma_{i}" for i in range(12)]
        + [f"mel_{i}" for i in range(128)]
    )

In [22]:
# ============================================================
# 4. PROCESS RAVDESS FILES
# ============================================================

def process_dataset():
    """
    Process all RAVDESS audio files.

    For each audio file:
    1. Read metadata from the filename
    2. Extract audio features
    3. Validate the extracted features
    4. Store metadata and features in a dataframe

    Returns:
        features_df: Processed feature dataframe
        corrupt_files: List of files that could not be processed
    """

    records = []
    corrupt_files = []

    # Find all RAVDESS audio files
    files = glob.glob(
        os.path.join(DATA_PATH, "Actor_*", "*.wav")
    )

    if not files:
        raise FileNotFoundError(
            f"No .wav files found under {DATA_PATH}"
        )

    print(f"Found {len(files)} audio files.")

    # Process each audio file
    for file in tqdm(files, desc="Extracting features"):

        file_name = os.path.basename(file)

        try:
            # ------------------------------------------------
            # Extract metadata from filename
            # RAVDESS format:
            # 03-01-05-01-02-02-12-01.wav
            #              ↑
            #            Actor
            # ------------------------------------------------

            parts = file_name.split("-")

            if len(parts) != 7:
                raise ValueError("Invalid RAVDESS filename format")

            emotion_code = parts[2]
            actor_id = parts[6].replace(".wav", "")

            # Check whether emotion code is valid
            if emotion_code not in EMOTION_MAP:
                raise ValueError(
                    f"Unknown emotion code: {emotion_code}"
                )

            emotion = EMOTION_MAP[emotion_code]

            # ------------------------------------------------
            # Extract audio features
            # ------------------------------------------------

            feature = extract_features(file)

            # ------------------------------------------------
            # Feature validation
            # Expected:
            # 40 MFCC + 12 Chroma + 128 Mel = 180 features
            # ------------------------------------------------

            if feature.shape[0] != 180:
                raise ValueError(
                    f"Expected 180 features, got {feature.shape[0]}"
                )

            if np.isnan(feature).any():
                raise ValueError("NaN values found in features")

            if np.isinf(feature).any():
                raise ValueError("Infinite values found in features")

            # ------------------------------------------------
            # Store metadata + features
            # ------------------------------------------------

            record = {
                "file_path": file,
                "actor_id": actor_id,
                "emotion": emotion
            }

            # Add feature values
            feature_names = get_feature_names()

            for name, value in zip(feature_names, feature):
                record[name] = value

            records.append(record)

        except Exception as e:

            corrupt_files.append(file)

            print(
                f"[WARN] Skipping {file}: {e}"
            )

    # Create dataframe
    features_df = pd.DataFrame(records)

    return features_df, corrupt_files

In [23]:
# ============================================================
# FEATURE COLUMN NAMES
# ============================================================

def get_feature_names():
    """
    Return column names for all extracted audio features.

    Total:
    40 MFCC + 12 Chroma + 128 Mel = 180 features
    """

    return (
        [f"mfcc_{i}" for i in range(40)]
        + [f"chroma_{i}" for i in range(12)]
        + [f"mel_{i}" for i in range(128)]
    )

In [24]:
def load_data():
    x, y, file_log = [], [], []
    corrupt_files = []

    files = glob.glob(
        os.path.join(DATA_PATH, "Actor_*", "*.wav")
    )

    if not files:
        raise FileNotFoundError(
            f"No .wav files found under {DATA_PATH}"
        )

    for file in tqdm(files, desc="Extracting features"):

        file_name = os.path.basename(file)
        parts = file_name.split("-")

        try:
            # Extract metadata from RAVDESS filename
            emotion_code = parts[2]
            actor_id = parts[6].replace(".wav", "")

            emotion = EMOTION_MAP[emotion_code]

            # Extract audio features
            feature = extract_features(file)

            # Schema validation:
            # 40 MFCC + 12 Chroma + 128 Mel = 180 features
            if (
                feature.shape[0] != 180
                or np.isnan(feature).any()
                or np.isinf(feature).any()
            ):
                corrupt_files.append(file)
                continue

            # Store features
            x.append(feature)

            # Store metadata
            y.append(emotion)

            file_log.append({
                "file": file,
                "actor_id": actor_id,
                "emotion": emotion
            })

        except Exception as e:
            corrupt_files.append(file)

            print(
                f"[WARN] Skipping {file}: {e}"
            )

    # Convert extracted features to NumPy array
    X = np.array(x)

    # Create feature column names
    feature_names = get_feature_names()

    # Create dataframe
    features_df = pd.DataFrame(
        X,
        columns=feature_names
    )

    # Add metadata
    log_df = pd.DataFrame(file_log)

    features_df.insert(
        0,
        "emotion",
        log_df["emotion"].values
    )

    features_df.insert(
        0,
        "actor_id",
        log_df["actor_id"].values
    )

    features_df.insert(
        0,
        "file_path",
        log_df["file"].values
    )

    return features_df, corrupt_files

In [25]:
# ============================================================
# 7. DATA QUALITY LOG & AUDIT
# ============================================================

def write_quality_log(features_df, corrupt_files, elapsed_s):
    """
    Create a data quality audit log for the preprocessing stage.

    Checks:
    - Number of raw files processed
    - Number of corrupt/skipped files
    - Expected file count
    - Missing values
    - Infinite values
    - Duplicate rows
    - Duplicate file paths
    - Feature dimensions
    - Number of actors
    - Emotion distribution
    """

    feature_columns = get_feature_names()

    lines = []

    lines.append("DATA QUALITY LOG & AUDIT SHEET")
    lines.append("=" * 45)

    # --------------------------------------------------------
    # Basic dataset information
    # --------------------------------------------------------

    lines.append(f"Source path          : {DATA_PATH}")
    lines.append(f"Extraction time      : {elapsed_s:.1f}s")

    lines.append(f"Total usable files   : {len(features_df)}")
    lines.append(f"Corrupt/skipped      : {len(corrupt_files)}")

    # --------------------------------------------------------
    # Feature information
    # --------------------------------------------------------

    lines.append(
        f"Feature dimensions   : {len(feature_columns)} "
        f"(40 MFCC + 12 Chroma + 128 Mel)"
    )

    # --------------------------------------------------------
    # Missing values
    # --------------------------------------------------------

    missing_values = (
        features_df[feature_columns]
        .isna()
        .sum()
        .sum()
    )

    lines.append(
        f"Missing feature values : {int(missing_values)}"
    )

    # --------------------------------------------------------
    # Infinite values
    # --------------------------------------------------------

    infinite_values = np.isinf(
        features_df[feature_columns].values
    ).sum()

    lines.append(
        f"Infinite feature values: {int(infinite_values)}"
    )

    # --------------------------------------------------------
    # Duplicate checks
    # --------------------------------------------------------

    duplicate_rows = features_df.duplicated().sum()

    duplicate_files = (
        features_df["file_path"]
        .duplicated()
        .sum()
    )

    lines.append(
        f"Duplicate rows        : {int(duplicate_rows)}"
    )

    lines.append(
        f"Duplicate file paths  : {int(duplicate_files)}"
    )

    # --------------------------------------------------------
    # Actor information
    # --------------------------------------------------------

    unique_actors = features_df["actor_id"].nunique()

    lines.append(
        f"Unique actors         : {unique_actors}"
    )

    # --------------------------------------------------------
    # Emotion distribution
    # --------------------------------------------------------

    lines.append("")
    lines.append("Emotion distribution:")

    emotion_counts = (
        features_df["emotion"]
        .value_counts()
        .sort_index()
    )

    for emotion, count in emotion_counts.items():
        lines.append(
            f"  {emotion:15s} {count}"
        )

    # --------------------------------------------------------
    # Preprocessing validation
    # --------------------------------------------------------

    lines.append("")
    lines.append("Preprocessing validation:")

    # RAVDESS contains 1,440 audio files
    expected_files = 1440

    # Check whether all expected files were processed
    if (
        len(features_df) == expected_files
        and len(corrupt_files) == 0
    ):
        lines.append(
            "  File count check    : PASSED"
        )
    else:
        lines.append(
            f"  File count check    : REVIEW "
            f"(expected {expected_files}, "
            f"got {len(features_df)})"
        )

    # Overall preprocessing status
    if (
        len(features_df) == expected_files
        and len(corrupt_files) == 0
        and missing_values == 0
        and infinite_values == 0
        and duplicate_rows == 0
        and duplicate_files == 0
    ):
        lines.append("  STATUS: PASSED")
    else:
        lines.append("  STATUS: REVIEW REQUIRED")

    # --------------------------------------------------------
    # Save audit log
    # --------------------------------------------------------

    log_text = "\n".join(lines)

    log_file = os.path.join(
        OUTPUT_PATH,
        "data_quality_log.txt"
    )

    with open(log_file, "w") as f:
        f.write(log_text)

    print(log_text)
    print(f"\nAudit log saved to: {log_file}")

In [26]:
# ============================================================
# 8. MAIN PREPROCESSING PIPELINE
# ============================================================

import time


def main():

    # Start timer to measure preprocessing time
    t0 = time.time()

    # --------------------------------------------------------
    # Extract features and metadata from raw audio
    # --------------------------------------------------------

    features_df, corrupt_files = load_data()

    elapsed = time.time() - t0

    # --------------------------------------------------------
    # Save list of corrupt/skipped files
    # --------------------------------------------------------

    if corrupt_files:
        pd.DataFrame(
            {"corrupt_file": corrupt_files}
        ).to_csv(
            os.path.join(
                OUTPUT_PATH,
                "corrupt_files.csv"
            ),
            index=False
        )

    # --------------------------------------------------------
    # Save processed feature dataset
    # --------------------------------------------------------

    features_file = os.path.join(
        OUTPUT_PATH,
        "features.csv"
    )

    features_df.to_csv(
        features_file,
        index=False
    )

    # --------------------------------------------------------
    # Generate data quality audit
    # --------------------------------------------------------

    write_quality_log(
        features_df,
        corrupt_files,
        elapsed
    )

    # --------------------------------------------------------
    # Final summary
    # --------------------------------------------------------

    print("\n========================================")
    print("PREPROCESSING COMPLETE")
    print("========================================")

    print(f"Valid files       : {len(features_df)}")
    print(f"Corrupt files     : {len(corrupt_files)}")
    print(f"Feature columns   : {len(get_feature_names())}")
    print(f"Processing time   : {elapsed:.1f} seconds")
    print(f"Feature CSV       : {features_file}")


if __name__ == "__main__":
    main()

Extracting features: 100%|██████████| 1440/1440 [00:43<00:00, 33.44it/s]


DATA QUALITY LOG & AUDIT SHEET
Source path          : D:\SJU\ML assn\Audio_Speech_Actors_01-24_16k
Extraction time      : 43.1s
Total usable files   : 1440
Corrupt/skipped      : 0
Feature dimensions   : 180 (40 MFCC + 12 Chroma + 128 Mel)
Missing feature values : 0
Infinite feature values: 0
Duplicate rows        : 0
Duplicate file paths  : 0
Unique actors         : 24

Emotion distribution:
  angry           192
  calm            192
  disgust         192
  fearful         192
  happy           192
  neutral         96
  sad             192
  surprised       192

Preprocessing validation:
  File count check    : PASSED
  STATUS: PASSED

Audit log saved to: D:\SJU\ML assn\CallConnect\data\processed\data_quality_log.txt

PREPROCESSING COMPLETE
Valid files       : 1440
Corrupt files     : 0
Feature columns   : 180
Processing time   : 43.1 seconds
Feature CSV       : D:\SJU\ML assn\CallConnect\data\processed\features.csv


In [27]:
# ============================================================
# FINAL CSV VALIDATION
# ============================================================

saved_df = pd.read_csv(
    os.path.join(OUTPUT_PATH, "features.csv")
)

print("Saved CSV shape:", saved_df.shape)
print("Missing values:", saved_df.isnull().sum().sum())
print("Duplicate rows:", saved_df.duplicated().sum())
print("Columns:", len(saved_df.columns))

Saved CSV shape: (1440, 183)
Missing values: 0
Duplicate rows: 0
Columns: 183
